# Experiment with trained UNet Model

In [ ]:
# Import necessary libraries
from unet_model import UNet
import torch
from torchvision import transforms as tfs
import os
from PIL import Image
import numpy as np

In [ ]:
input_dir = './inputs'
input_image_name = 'image2.png'
output_dir = './outputs'
output_image_name = 'output_image.jpeg'
best_model_path = os.path.join(output_dir, 'best_model.pth')

In [ ]:
# Check if a GPU is available and set the device accordingly
device = "cpu"
# Check if a GPU is available and set the device accordingly
if torch.accelerator.is_available():
    device = torch.accelerator.current_accelerator().type

print(f"Using {device} device")

In [ ]:
# Initialize the UNet model and move it to the appropriate device
model = UNet().to(device)

In [ ]:
# Load model from checkpoint
if os.path.exists(best_model_path):
    model.load_state_dict(torch.load(best_model_path))
    print(f"Loaded best model from {best_model_path}")
else:
    print(f"Best model not found at {best_model_path}.")

In [ ]:
# Define transformations (same as training)
transform = tfs.Compose([
    tfs.Resize((256, 256)),
    tfs.ToTensor(),
    tfs.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Denormalization function
def denormalize(tensor, mean, std):
    mean = torch.tensor(mean).view(3, 1, 1)
    std = torch.tensor(std).view(3, 1, 1)
    return tensor * std + mean

In [ ]:
# Test the model for an input image located in inputs directory and save the output
def test_model(input_image_path, output_image_path):
    model.eval()
    with torch.no_grad():
        # Load and preprocess the input image
        input_image_path = os.path.join(input_dir, input_image_name)
        input_image = Image.open(input_image_path).convert('RGB')  # Use Pillow to load the image
        input_tensor = transform(input_image).unsqueeze(0).to(device)

        # Forward pass through the model
        output_tensor = model(input_tensor)

        # Denormalize the input and output images
        mean = [0.5, 0.5, 0.5]
        std = [0.5, 0.5, 0.5]
        input_image_np = np.array(input_image.resize((output_tensor.shape[3], output_tensor.shape[2]))) / 255.0  # Resize and normalize input image
        output_image_np = denormalize(output_tensor.squeeze(0).cpu(), mean, std).permute(1, 2, 0).numpy().clip(0, 1)

        # Combine input and output images side-by-side
        comparison_image = np.concatenate((input_image_np, output_image_np), axis=1)

        # Convert to uint8 and save
        comparison_image = (comparison_image * 255).astype(np.uint8)
        comparison_pil = Image.fromarray(comparison_image)
        output_image_path = os.path.join(output_dir, output_image_name)
        comparison_pil.save(output_image_path)
        print(f"Saved comparison image to {output_image_path}")

In [ ]:
# Run the test
test_model(input_image_name, output_image_name)